# Complete RAG pipeline

This notebook loads PDFs and text files, chunks them, embeds them with SentenceTransformers, stores them in a persistent ChromaDB collection, retrieves relevant chunks, and returns an answer with citations. If `MISTRAL_API_KEY` is available in the project `.env` file and `langchain-mistralai` is installed, it can generate Mistral AI answers; otherwise it falls back to an extractive answer from the retrieved context.

In [1]:
from __future__ import annotations

import hashlib
import os
import shutil
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional

import chromadb
import numpy as np
from sentence_transformers import SentenceTransformer

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

try:
    from pypdf import PdfReader
except ImportError:
    PdfReader = None

try:
    import fitz  # PyMuPDF, used as a fallback PDF reader
except ImportError:
    fitz = None


d:\NLP\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Project paths
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == "notebook" else NOTEBOOK_DIR
DATA_DIR = PROJECT_ROOT / "data"
PDF_DIR = DATA_DIR / "pdfs"
TEXT_DIR = DATA_DIR / "text_files"
VECTOR_DIR = DATA_DIR / "vector_store"

for directory in (DATA_DIR, PDF_DIR, TEXT_DIR, VECTOR_DIR):
    directory.mkdir(parents=True, exist_ok=True)

ENV_PATH = PROJECT_ROOT / ".env"
if load_dotenv is not None and ENV_PATH.exists():
    load_dotenv(ENV_PATH)
    print(f"Loaded environment variables from: {ENV_PATH}")
elif ENV_PATH.exists():
    print("Install python-dotenv to load .env automatically, or set MISTRAL_API_KEY in your environment.")

print(f"Project root: {PROJECT_ROOT}")
print(f"PDF folder:   {PDF_DIR}")
print(f"Text folder:  {TEXT_DIR}")
print(f"Vector store: {VECTOR_DIR}")


Loaded environment variables from: d:\NLP\RAG\.env
Project root: d:\NLP\RAG
PDF folder:   d:\NLP\RAG\data\pdfs
Text folder:  d:\NLP\RAG\data\text_files
Vector store: d:\NLP\RAG\data\vector_store


In [3]:
@dataclass
class Document:
    page_content: str
    metadata: Dict[str, Any]


def _clean_text(text: str) -> str:
    """Normalize whitespace while preserving paragraph boundaries enough for retrieval."""
    lines = [line.strip() for line in text.replace("\x00", " ").splitlines()]
    paragraphs: List[str] = []
    buffer: List[str] = []

    for line in lines:
        if line:
            buffer.append(line)
        elif buffer:
            paragraphs.append(" ".join(buffer))
            buffer = []

    if buffer:
        paragraphs.append(" ".join(buffer))

    return "\n\n".join(paragraphs).strip()


def load_pdf(path: Path) -> List[Document]:
    """Load one PDF into page-level documents."""
    documents: List[Document] = []

    if PdfReader is not None:
        reader = PdfReader(str(path))
        for page_number, page in enumerate(reader.pages, start=1):
            text = _clean_text(page.extract_text() or "")
            if text:
                documents.append(
                    Document(
                        page_content=text,
                        metadata={
                            "source": str(path),
                            "source_name": path.name,
                            "source_type": "pdf",
                            "page": page_number,
                        },
                    )
                )
        return documents

    if fitz is not None:
        with fitz.open(path) as pdf:
            for page_index in range(pdf.page_count):
                page = pdf.load_page(page_index)
                text = _clean_text(page.get_text("text") or "")
                if text:
                    documents.append(
                        Document(
                            page_content=text,
                            metadata={
                                "source": str(path),
                                "source_name": path.name,
                                "source_type": "pdf",
                                "page": page_index + 1,
                            },
                        )
                    )
        return documents

    raise ImportError("Install pypdf or pymupdf to load PDF files.")


def load_text_file(path: Path) -> List[Document]:
    text = _clean_text(path.read_text(encoding="utf-8", errors="ignore"))
    if not text:
        return []
    return [
        Document(
            page_content=text,
            metadata={
                "source": str(path),
                "source_name": path.name,
                "source_type": path.suffix.lower().lstrip(".") or "text",
            },
        )
    ]


def load_documents(input_paths: Optional[Iterable[Path | str]] = None) -> List[Document]:
    """Load PDFs, .txt, and .md files from the project data folders or explicit paths."""
    if input_paths is None:
        paths = sorted(PDF_DIR.glob("*.pdf")) + sorted(TEXT_DIR.glob("*.txt")) + sorted(TEXT_DIR.glob("*.md"))
    else:
        paths = [Path(p) for p in input_paths]

    documents: List[Document] = []
    for path in paths:
        if not path.exists():
            print(f"Skipping missing path: {path}")
            continue
        if path.is_dir():
            documents.extend(load_documents(path.rglob("*")))
            continue
        suffix = path.suffix.lower()
        if suffix == ".pdf":
            documents.extend(load_pdf(path))
        elif suffix in {".txt", ".md"}:
            documents.extend(load_text_file(path))

    print(f"Loaded {len(documents)} source document/page(s).")
    return documents

In [4]:
def split_documents(
    documents: List[Document],
    chunk_size: int = 900,
    chunk_overlap: int = 150,
) -> List[Document]:
    """Split documents into overlapping character chunks."""
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")
    if not 0 <= chunk_overlap < chunk_size:
        raise ValueError("chunk_overlap must be >= 0 and smaller than chunk_size")

    chunks: List[Document] = []
    step = chunk_size - chunk_overlap

    for doc in documents:
        text = doc.page_content.strip()
        if not text:
            continue

        start = 0
        chunk_index = 0
        while start < len(text):
            end = min(start + chunk_size, len(text))

            # Prefer ending at a paragraph or sentence boundary when nearby.
            if end < len(text):
                boundary = max(text.rfind("\n\n", start, end), text.rfind(". ", start, end))
                if boundary > start + int(chunk_size * 0.55):
                    end = boundary + (2 if text[boundary : boundary + 2] == "\n\n" else 1)

            chunk_text = text[start:end].strip()
            if chunk_text:
                metadata = dict(doc.metadata)
                metadata["chunk_index"] = chunk_index
                metadata["chunk_start"] = start
                metadata["chunk_end"] = end
                chunks.append(Document(page_content=chunk_text, metadata=metadata))
                chunk_index += 1

            if end >= len(text):
                break
            next_start = max(0, end - chunk_overlap)
            if next_start <= start:
                next_start = start + step
            start = next_start

    print(f"Created {len(chunks)} chunk(s).")
    return chunks


In [5]:
class EmbeddingManager:
    """Small wrapper around SentenceTransformers with defensive input checks."""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = SentenceTransformer(model_name)
        if hasattr(self.model, "get_embedding_dimension"):
            self.dimension = self.model.get_embedding_dimension()
        else:
            self.dimension = self.model.get_sentence_embedding_dimension()
        print(f"Loaded embedding model: {model_name} ({self.dimension} dimensions)")

    @staticmethod
    def _clean_embedding_text(text: Any, label: str = "text") -> Optional[str]:
        if not isinstance(text, str):
            print(f"Skipping {label}: expected text, got {type(text).__name__}.")
            return None

        cleaned = text.replace("\x00", " ").strip()
        if not cleaned:
            print(f"Skipping {label}: empty text.")
            return None
        return cleaned

    def _encode_batch(self, texts: List[str], batch_size: int = 32) -> np.ndarray:
        embeddings = self.model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=len(texts) > batch_size,
            normalize_embeddings=True,
        )
        embeddings = np.asarray(embeddings, dtype=np.float32)
        return np.atleast_2d(embeddings)

    def generate_embeddings(self, texts: List[str], batch_size: int = 32) -> np.ndarray:
        clean_texts = []
        for index, text in enumerate(texts):
            cleaned = self._clean_embedding_text(text, label=f"input {index}")
            if cleaned is not None:
                clean_texts.append(cleaned)

        if not clean_texts:
            return np.empty((0, self.dimension), dtype=np.float32)

        try:
            return self._encode_batch(clean_texts, batch_size=batch_size)
        except Exception as exc:
            print(f"Embedding batch failed; retrying one item at a time and skipping failures. Reason: {exc}")

        rows = []
        for index, text in enumerate(clean_texts):
            try:
                rows.append(self._encode_batch([text], batch_size=1)[0])
            except Exception as exc:
                preview = text[:80].replace("\n", " ")
                print(f"Skipping input {index}: embedding failed ({exc}). Preview: {preview!r}")

        if not rows:
            return np.empty((0, self.dimension), dtype=np.float32)
        return np.vstack(rows).astype(np.float32)

    def generate_document_embeddings(
        self,
        documents: List[Document],
        batch_size: int = 32,
    ) -> tuple[List[Document], np.ndarray]:
        clean_pairs = []
        for index, document in enumerate(documents):
            cleaned = self._clean_embedding_text(document.page_content, label=f"chunk {index}")
            if cleaned is None:
                continue
            document.page_content = cleaned
            clean_pairs.append((document, cleaned))

        if not clean_pairs:
            return [], np.empty((0, self.dimension), dtype=np.float32)

        clean_documents = [document for document, _ in clean_pairs]
        clean_texts = [text for _, text in clean_pairs]

        try:
            return clean_documents, self._encode_batch(clean_texts, batch_size=batch_size)
        except Exception as exc:
            print(f"Embedding batch failed; retrying chunks individually and skipping bad chunks. Reason: {exc}")

        kept_documents = []
        rows = []
        for index, (document, text) in enumerate(clean_pairs):
            try:
                rows.append(self._encode_batch([text], batch_size=1)[0])
                kept_documents.append(document)
            except Exception as exc:
                source = document.metadata.get("source_name", document.metadata.get("source", "unknown"))
                preview = text[:80].replace("\n", " ")
                print(f"Skipping chunk {index} from {source}: embedding failed ({exc}). Preview: {preview!r}")

        if not rows:
            return [], np.empty((0, self.dimension), dtype=np.float32)
        return kept_documents, np.vstack(rows).astype(np.float32)


In [6]:
class VectorStore:
    """Persistent Chroma vector store with deterministic IDs to avoid duplicate inserts."""

    def __init__(
        self,
        collection_name: str = "rag_documents",
        persist_directory: Path | str = VECTOR_DIR,
        reset: bool = False,
    ):
        self.collection_name = collection_name
        self.persist_directory = Path(persist_directory)

        if reset and self.persist_directory.exists():
            shutil.rmtree(self.persist_directory)

        self.persist_directory.mkdir(parents=True, exist_ok=True)
        self.client = chromadb.PersistentClient(path=str(self.persist_directory))
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            metadata={"description": "Documents and chunks for local RAG"},
        )
        print(f"Collection: {collection_name}; existing chunks: {self.collection.count()}")

    @staticmethod
    def _document_id(doc: Document) -> str:
        source = doc.metadata.get("source", "")
        page = doc.metadata.get("page", "")
        chunk_index = doc.metadata.get("chunk_index", "")
        digest = hashlib.sha256(f"{source}|{page}|{chunk_index}|{doc.page_content}".encode("utf-8")).hexdigest()
        return f"chunk_{digest[:24]}"

    def add_documents(self, documents: List[Document], embeddings: np.ndarray, upsert: bool = True) -> None:
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        if not documents:
            print("No documents to add.")
            return

        ids = [self._document_id(doc) for doc in documents]
        metadatas = [dict(doc.metadata) for doc in documents]
        texts = [doc.page_content for doc in documents]
        vectors = [embedding.tolist() for embedding in embeddings]

        method = self.collection.upsert if upsert else self.collection.add
        method(ids=ids, embeddings=vectors, metadatas=metadatas, documents=texts)
        print(f"Indexed {len(documents)} chunk(s). Collection now has {self.collection.count()} chunk(s).")

    def clear(self) -> None:
        self.client.delete_collection(self.collection_name)
        self.collection = self.client.get_or_create_collection(name=self.collection_name)
        print(f"Cleared collection: {self.collection_name}")


In [7]:
class RAGRetriever:
    """Query Chroma and return ranked chunks with relevance scores."""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    @staticmethod
    def _distance_to_score(distance: Optional[float]) -> float:
        if distance is None:
            return 0.0
        # Chroma cosine distance is usually 0 for identical and larger for less similar.
        return max(0.0, min(1.0, 1.0 - float(distance)))

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        query = query.strip() if isinstance(query, str) else ""
        if not query:
            print("Skipping retrieval: query is empty.")
            return []
        if self.vector_store.collection.count() == 0:
            print("Skipping retrieval: vector store is empty. Run rag.index_documents() first.")
            return []

        query_embeddings = self.embedding_manager.generate_embeddings([query])
        if len(query_embeddings) == 0:
            print("Skipping retrieval: query embedding could not be generated.")
            return []

        try:
            raw = self.vector_store.collection.query(
                query_embeddings=[query_embeddings[0].tolist()],
                n_results=top_k,
                include=["documents", "metadatas", "distances"],
            )
        except Exception as exc:
            print(f"Skipping retrieval: Chroma query failed. Reason: {exc}")
            return []

        documents = raw.get("documents", [[]])[0]
        metadatas = raw.get("metadatas", [[]])[0]
        distances = raw.get("distances", [[]])[0]
        ids = raw.get("ids", [[]])[0]

        results: List[Dict[str, Any]] = []
        for rank, (doc_id, text, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances), start=1):
            score = self._distance_to_score(distance)
            if score < score_threshold:
                continue
            results.append(
                {
                    "rank": rank,
                    "id": doc_id,
                    "text": text,
                    "metadata": metadata or {},
                    "distance": distance,
                    "score": score,
                }
            )

        return results


In [8]:
class RAGPipeline:
    """End-to-end local RAG pipeline."""

    def __init__(
        self,
        embedding_model: str = "all-MiniLM-L6-v2",
        collection_name: str = "rag_documents",
        persist_directory: Path | str = VECTOR_DIR,
    ):
        self.embedding_manager = EmbeddingManager(embedding_model)
        self.vector_store = VectorStore(collection_name=collection_name, persist_directory=persist_directory)
        self.retriever = RAGRetriever(self.vector_store, self.embedding_manager)

    def index_documents(
        self,
        input_paths: Optional[Iterable[Path | str]] = None,
        chunk_size: int = 900,
        chunk_overlap: int = 150,
        reset: bool = False,
    ) -> List[Document]:
        if reset:
            self.vector_store.clear()

        documents = load_documents(input_paths)
        chunks = split_documents(documents, chunk_size=chunk_size, chunk_overlap=chunk_overlap)
        chunks, embeddings = self.embedding_manager.generate_document_embeddings(chunks)
        if not chunks or len(embeddings) == 0:
            print("No embeddable chunks found; nothing was indexed.")
            return []

        self.vector_store.add_documents(chunks, embeddings, upsert=True)
        return chunks

    def ask(self, query: str, top_k: int = 5, score_threshold: float = 0.0, use_llm: bool = False) -> Dict[str, Any]:
        try:
            contexts = self.retriever.retrieve(query, top_k=top_k, score_threshold=score_threshold)
        except Exception as exc:
            print(f"Retrieval failed; returning fallback answer. Reason: {exc}")
            contexts = []
        answer = generate_answer(query, contexts, use_llm=use_llm)
        return {"query": query, "answer": answer, "contexts": contexts}


In [9]:
def format_citations(contexts: List[Dict[str, Any]]) -> str:
    lines = []
    for item in contexts:
        meta = item["metadata"]
        page = f", page {meta['page']}" if meta.get("page") else ""
        lines.append(
            f"[{item['rank']}] {meta.get('source_name', meta.get('source', 'unknown'))}{page} "
            f"(score={item['score']:.3f})"
        )
    return "\n".join(lines)


def build_prompt(query: str, contexts: List[Dict[str, Any]]) -> str:
    context_text = "\n\n".join(
        f"[{item['rank']}] Source: {item['metadata'].get('source_name', 'unknown')}\n{item['text']}"
        for item in contexts
    )
    
    return f"""Answer the question using only the context below. If the context is insufficient, say so.

Question:
{query}

Context:
{context_text}

Answer with concise citations like [1], [2] where useful.
"""
def generate_answer(query: str, contexts: List[Dict[str, Any]], use_llm: bool = False) -> str:
    if not contexts:
        return "I could not find relevant context in the indexed documents."

    if use_llm:
        try:
            from langchain_mistralai import ChatMistralAI
            from langchain_core.messages import HumanMessage

            if not os.getenv("MISTRAL_API_KEY"):
                raise ValueError("MISTRAL_API_KEY was not found. Add it to the project .env file or environment.")

            llm = ChatMistralAI(
                model=os.getenv("MISTRAL_MODEL", "mistral-small-latest"),
                temperature=0,
            )
            response = llm.invoke([HumanMessage(content=build_prompt(query, contexts))])
            return response.content
        except Exception as exc:
            print(f"Mistral generation failed; using extractive fallback. Reason: {exc}")

    best = contexts[0]
    answer = best["text"]
    if len(answer) > 1200:
        answer = answer[:1200].rsplit(" ", 1)[0] + "..."
    return f"{answer}\n\nSources:\n{format_citations(contexts)}"


## Build or update the index

Put PDFs in `data/pdfs/` and text/Markdown files in `data/text_files/`, then run the next cell. Use `reset=True` when you want to rebuild the collection from scratch.


In [10]:
rag = RAGPipeline()

# Index every supported file under data/pdfs and data/text_files.
# Use reset=True if old chunks should be removed before indexing.
chunks = rag.index_documents(reset=False)
len(chunks)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14716.35it/s]


Loaded embedding model: all-MiniLM-L6-v2 (384 dimensions)
Collection: rag_documents; existing chunks: 960
Loaded 3 source document/page(s).
Created 7 chunk(s).
Indexed 7 chunk(s). Collection now has 960 chunk(s).


7

## Ask questions

Set `use_llm=True` to generate the final answer with Mistral AI. The notebook automatically loads `MISTRAL_API_KEY` from `D:\\NLP\\RAG\\.env`; you can optionally set `MISTRAL_MODEL` there too. Without Mistral generation, the notebook still works and returns the highest-ranked context with source citations.


In [11]:
prompt = input("Enter the prompt: ")
result = rag.ask(f"{prompt}", top_k=3, score_threshold=0.0, use_llm=True)
print(result["answer"])


Insufficient context to answer the question.


In [12]:
# Inspect raw retrieved chunks when debugging retrieval quality.
for ctx in result["contexts"]:
    print(f"Rank {ctx['rank']} | score={ctx['score']:.3f} | {ctx['metadata'].get('source_name')}")
    print(ctx["text"][:500])
    print("-" * 80)


Rank 1 | score=0.000 | algoPython.pdf
'units': '-100000.0', 'price': 1.19029, 'guaranteedExecutionFee': '0.0', 'halfSpreadCost': '5.4606', 'initialMarginRequired': '3330.0'}, 'tradesClosed': [{'tradeID': '1735', 'units': '-100000.0', 'price': 1.19029, 'realizedPL': '-0.8443', 'financing': '0.0', 'guaranteedExecutionFee': '0.0', 'halfSpreadCost': '5.4606'}], 'halfSpreadCost': '10.9212'} *** GOING LONG *** {'id': '1739', 'time': '2020-08-19T14:48:15.529680632Z', 'userID': 13834683, 'accountID': '101-004-13834683-001', 'batchID': '1738
--------------------------------------------------------------------------------
Rank 2 | score=0.000 | algoPython.pdf
s': [{'price': 1.1911, 'liquidity': '10000000'}], 'asks': [{'price': 1.19131, 'liquidity': '10000000'}], 'closeoutBid': 1.1911, 'closeoutAsk': 1.19131}, 'reason': 'MARKET_ORDER', 'pl': '0.0', 'financing': '0.0', Placing Market Orders | 237
--------------------------------------------------------------------------------
Rank 3 | score=0.000 

In [13]:
import os

print(os.getenv("MISTRAL_API_KEY")[:10])

1w30Tin8TV
